# Imports and Config

In [ ]:
import os
import pickle
from grf_pipeline_utils.signal_processing import *
from grf_pipeline_utils.opensim_utils import *
import yaml

repo_root = os.path.abspath('../')
with open(os.path.join(repo_root, 'config.yaml')) as f:
    cfg = yaml.safe_load(f)

root_dir        = os.path.join(repo_root, cfg['silder']['data_root'])
transformed_dir = os.path.join(repo_root, cfg['silder']['results']['transformed'])
processed_dir   = os.path.join(repo_root, cfg['silder']['results']['processed'])
scaling_dir     = os.path.join(repo_root, cfg['silder']['results']['scaling'])
ik_dir          = os.path.join(repo_root, cfg['silder']['results']['ik'])
id_dir          = os.path.join(repo_root, cfg['silder']['results']['id_raw'])

OA_subjects  = [f"OA{i}" for i in cfg['silder']['OA_subjects']]
Y_subjects   = [f"Y{i}"  for i in cfg['silder']['Y_subjects']]

speeds       = cfg['silder']['speeds']
n_trials     = cfg['silder']['n_trials']
subj_masses  = {**cfg['silder']['OA_masses'], **cfg['silder']['Y_masses']}
_sv_major = cfg['active']['silder_version'].split('.')[0]


# Build Subject_Trials Dictionary

In [ ]:
subjects = OA_subjects + Y_subjects
trial_names = []

subject_trials = {}
for subj in subjects:
    subj_dir = os.path.join(root_dir, subj, 'Walking/Files_W_HJCs/')
    # OA subjects use '_walk_static1' while Y subjects use '_walking_static1'
    static_name = f'{subj}_walk_static1.trc' if subj[0] == 'O' else f'{subj}_walking_static1.trc'
    subject_trials[subj] = {
        'static': {
            'input':  os.path.join(subj_dir, static_name),
            'output': os.path.join(transformed_dir, f'{subj}_walk_static1_transformed.trc')
        },
        'tracking': [],
        'forces': []
    }
    for spd in speeds:
        for i in range(1, n_trials + 1):
            trial_name = f'{subj}_{spd}_{i}'
            trial_names.append(trial_name)
            subject_trials[subj]['tracking'].append({
                'input':  os.path.join(subj_dir, f'{trial_name}.trc'),
                'output': os.path.join(transformed_dir, f'{trial_name}_transformed.trc')
            })
            subject_trials[subj]['forces'].append({
                'input':  os.path.join(subj_dir, f'{trial_name}.forces'),
                'output': os.path.join(transformed_dir, f'{trial_name}_transformed.mot')
            })

# Preprocess Tracking and GRF Files

In [ ]:
all_segs = {}
for subj, data in subject_trials.items():
    process_hjc_trc(input_path=data['static']['input'],
                    output_path=data['static']['output'],
                    markers_to_drop=[])
    for trc, forces in zip(data['tracking'], data['forces']):
        trial_segs = preprocess_trc_grf(
            trc_ip=trc['input'],
            trc_op=trc['output'],
            markers_to_drop=[],
            grf_ip=forces['input'],
            grf_op=forces['output'],
            grf_pickle_path=os.path.join(processed_dir, 'grf_pickles')
        )
        for trial_name, seg_dict in trial_segs.items():
            subj_name = trial_name.split('_')[0]
            if subj_name not in all_segs:
                all_segs[subj_name] = {}
            all_segs[subj_name][trial_name] = seg_dict

with open(os.path.join(processed_dir, f'all_stance_segs_v{_sv_major}.pkl'), 'wb') as f:
    pickle.dump(all_segs, f)
print(f'Saved all_stance_segs_v{_sv_major}.pkl')

# Scaling

In [ ]:
for subj, data in subject_trials.items():
    scale_generic(
        root_dir=root_dir,
        mass=subj_masses[subj],
        static_pose_filename=data['static']['output'],
        scaling_dir=scaling_dir
    )

# Parse Scaling Log

In [ ]:
scaling_log = os.path.join(scaling_dir, 'Combined_scaling_log.txt')
parsed = parse_combined_scaling_output(scaling_log)
for subject, info in parsed.items():
    print(f"{subject} — RMS: {info['marker_error_rms']:.4f}, "
          f"max: {info['marker_error_max']:.4f} at {info['marker_error_max_marker']}")

Inverse kinematics too memory-intensive to run in notebook

# Parse IK Log

In [ ]:
ik_log = os.path.join(ik_dir, 'Combined_ik_log.txt')
ik_df = parse_full_ik_log(ik_log, trial_names)
problem_trials = []
mean_rms_count, mean_max_count = 0, 0

for idx, row in ik_df.iterrows():
    if row['mean_rms'] > 0.04:
        mean_rms_count += 1
        problem_trials.append(row['trial_name'])
    elif row['mean_max'] > 0.05:
        mean_max_count += 1
        problem_trials.append(row['trial_name'])

print(f'{mean_rms_count} trials with Mean RMS > 4 cm')
print(f'{mean_max_count} trials with Mean max error > 5 cm')
for t in problem_trials:
    print(t)

# Inverse Dynamics

In [ ]:
loads_dir       = os.path.join(repo_root, cfg['silder']['results']['loads'])
id_raw_dir      = os.path.join(repo_root, cfg['silder']['results']['id_raw'])
id_filtered_dir = os.path.join(repo_root, cfg['silder']['results']['id_filtered'])

for subj, data in subject_trials.items():
    model = osim.Model(os.path.join(scaling_dir, f'{subj}_scaled.osim'))
    for trc, forces in zip(data['tracking'], data['forces']):
        inverse_dynamics(
            root_dir=root_dir,
            force_data_filepath=forces['output'],
            tracking_data_filepath=trc['output'],
            model=model,
            loads_dir=loads_dir,
            id_raw_dir=id_raw_dir,
            id_filtered_dir=id_filtered_dir,
            setup_dir = os.path.join(repo_root, cfg['silder']['opensim_setup_dir'])        )